<!-- course-header -->
# Lesson 13: Embeddings and vector search

*Worked solutions for lesson 13. Read the lesson [online](https://ashutoshjoshi1.github.io/AI-ZerotoMastery/lessons/13-embeddings-vector-search.html) or [offline](../../lessons/13-embeddings-vector-search.html). The exercise version without solutions is `../13-embeddings-vector-search.ipynb`.*

<a href="https://colab.research.google.com/github/ashutoshjoshi1/AI-ZerotoMastery/blob/main/notebooks/solutions/13-embeddings-vector-search-solutions.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

Turn sentences and documents into vectors that capture meaning, search them by similarity from scratch and with FAISS or a vector database, and understand the approximate-nearest-neighbour indexes that make it fast at scale.

**Runs on:** Any computer (CPU). The first run downloads a small embedding model (about 90 MB); the ANN benchmark runs on synthetic data without any download.

**How to use it:** read a section of the lesson, then run the matching cells here, in order. Each exercise has a cell titled *Hands-on* with the same name as in the lesson. Change numbers, break things on purpose, and note what happens in your learning log. Cells that need an API key, a GPU or a download say so in their first line.

## Setup

Install what this lesson needs. Skip this cell if your environment from lesson 00 already has these packages.

In [ ]:
%pip install -q numpy scikit-learn sentence-transformers faiss-cpu chromadb

## Getting embeddings

**Option: Local model (sentence-transformers)**

In [ ]:
# Downloads a model or data on first run
# uv add sentence-transformers
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")   # small, fast, 384 dims; downloads once (~90 MB)
texts = [
    "The instrument stopped recording overnight.",
    "Data logging halted shortly after midnight.",
    "Calibrate the spectrometer before each campaign.",
    "What is the capital of France?",
]
E = model.encode(texts, normalize_embeddings=True)  # (4, 384), unit length
print(E.shape)
print((E @ E.T).round(2))                           # cosine similarity matrix

**Option: Hosted embedding API**

In [ ]:
# Needs an API key (see the setup cell)
from openai import OpenAI
import numpy as np

client = OpenAI()
r = client.embeddings.create(model="text-embedding-3-small", input=texts)
E = np.array([d.embedding for d in r.data])          # (4, 1536)
E /= np.linalg.norm(E, axis=1, keepdims=True)

In the similarity matrix from the local model, the first two sentences score far higher with each other than with the others, even though they share almost no words. That is the whole point.

### Choosing an embedding model

Quality on *your* data matters more than leaderboard rank: use the MTEB leaderboard for a shortlist, then let your own retrieval test decide. Also check:

- **Dimensions** cost memory and search time. Matryoshka models can be truncated (`truncate_dim` in sentence-transformers) with little loss; other models lose much more.
- **Maximum input length**: many small models read only 256 to 512 tokens and silently truncate the rest.
- **Query and document prefixes**: some models expect "query:" and "passage:". Read the model card, or use `encode_query` and `encode_document`.
- **Never mix models**: embeddings from different models live in different spaces. Change the model, re-embed everything.

### Hands-on: explore the similarity matrix

Build an intuition for what the geometry does and does not capture before you trust it in a search engine.

1. Write 12 sentences in four groups of three: paraphrases of one fact, same topic but different facts, unrelated, and negations ("the cooler is running" against "the cooler is not running").
2. Embed them with `all-MiniLM-L6-v2` and compute the full cosine matrix. Print, for each group, the mean similarity within the group and to the rest.
3. Find the highest-scoring pair that you would *not* call similar, and the lowest-scoring pair that you would.
4. You are done when you have the four group means and can state one kind of difference (negation is the usual one) that cosine similarity mostly ignores.

#### Solution: a solution

In [ ]:
# Downloads a model or data on first run
import numpy as np

groups = {
    "paraphrase": ["The desiccant must be changed twice a year.", "Replace the desiccant every six months.",
                   "Desiccant replacement is due semi-annually."],
    "same topic": ["The desiccant turns pink when saturated.", "Humid sites need desiccant more often.",
                   "The desiccant cartridge sits inside the enclosure."],
    "unrelated":  ["The tracker lost the sun this morning.", "Invoices are paid within thirty days.",
                   "Python lists are mutable."],
    "negation":   ["The cooler is running.", "The cooler is not running.", "The cooler has stopped running."],
}
sents = [s for g in groups.values() for s in g]
labels = [name for name, g in groups.items() for _ in g]
E = model.encode(sents, normalize_embeddings=True)
S = E @ E.T
for name in groups:
    inside = [S[i, j] for i in range(len(sents)) for j in range(len(sents)) if i < j and labels[i] == labels[j] == name]
    outside = [S[i, j] for i in range(len(sents)) for j in range(len(sents)) if labels[i] == name and labels[j] != name]
    print(f"{name:<11} within={np.mean(inside):.2f}  to others={np.mean(outside):.2f}")
pairs = sorted(((S[i, j], sents[i], sents[j]) for i in range(len(sents)) for j in range(i + 1, len(sents))), reverse=True)
print("most similar:", pairs[0])
print("least similar:", pairs[-1])

Paraphrases typically score 0.7 to 0.9 with each other, same-topic sentences 0.4 to 0.6, unrelated ones near 0, and the negation pair scores almost as high as a paraphrase: "running" and "not running" differ by one token that barely moves the vector. Embeddings capture *what a sentence is about* far better than *what it asserts*, which is why RAG hands the retrieved text to a model that can read it properly.

## Semantic search from scratch

With normalised vectors, search is a matrix multiplication and a sort. Do it yourself once before reaching for tools.

In [ ]:
# Downloads a model or data on first run
import numpy as np

docs = [
    "Replace the desiccant cartridge every six months in humid climates.",
    "If the tracker loses the sun, check the GPS time and the level of the head.",
    "Dark current increases with detector temperature; keep the cooler running.",
    "The filter wheel position error usually means a slipping belt.",
    "Upload data nightly over the cellular modem; retry on failure.",
    # ... in practice, thousands of chunks from manuals, logs and tickets
]
doc_vecs = model.encode(docs, normalize_embeddings=True)          # (N, 384)

def search(query: str, k: int = 3):
    q = model.encode([query], normalize_embeddings=True)[0]       # (384,)
    scores = doc_vecs @ q                                         # (N,) cosine similarities
    top = np.argsort(-scores)[:k]
    return [(round(float(scores[i]), 3), docs[i]) for i in top]

for hit in search("why is the noise in my spectra higher when it's hot?"):
    print(hit)
# the dark-current/temperature document ranks first, despite little word overlap

## FAISS and vector databases

### FAISS: a fast in-process index

FAISS (from Meta) is a library of highly optimised vector indexes. `IndexFlatIP` is exact inner-product search, the same maths as above, done fast.

In [ ]:
# Downloads a model or data on first run
# uv add faiss-cpu
import faiss

d = doc_vecs.shape[1]
index = faiss.IndexFlatIP(d)                 # exact search by inner product (= cosine for unit vectors)
index.add(doc_vecs.astype("float32"))
q = model.encode(["sun tracking problems"], normalize_embeddings=True).astype("float32")
scores, ids = index.search(q, 3)             # both (1, 3)
print([docs[i] for i in ids[0]])
faiss.write_index(index, "manuals.faiss")    # save; faiss.read_index to load

### Vector databases

In [ ]:
# Downloads a model or data on first run
# uv add chromadb
import chromadb

db = chromadb.PersistentClient(path="vector_db")
col = db.get_or_create_collection("manuals", metadata={"hnsw:space": "cosine"})
col.add(
    ids=[f"doc-{i}" for i in range(len(docs))],
    documents=docs,
    embeddings=doc_vecs.tolist(),
    metadatas=[{"source": "field-manual", "section": i} for i in range(len(docs))],
)
res = col.query(
    query_embeddings=model.encode(["belt slipping"], normalize_embeddings=True).tolist(),
    n_results=2,
    where={"source": "field-manual"},          # metadata filter
)
print(res["documents"])

```sql
-- the same idea in PostgreSQL with pgvector
CREATE EXTENSION IF NOT EXISTS vector;
CREATE TABLE chunks (id bigserial PRIMARY KEY, source text, body text, embedding vector(384));
CREATE INDEX ON chunks USING hnsw (embedding vector_cosine_ops);
SELECT body FROM chunks
WHERE source = 'field-manual'
ORDER BY embedding <=> $1      -- <=> is cosine distance; $1 is the query vector
LIMIT 5;
```

> **Start boring**
>
> If you already run PostgreSQL, pgvector is usually the right first choice: one database, transactions, SQL filters and backups you already understand. Move to a dedicated vector database when you have measured a need.

### Hands-on: search with a metadata filter

Filtering is where a vector database earns its keep, and it is worth seeing that it is nothing magic.

1. Extend `docs` to about 15 short maintenance notes with a metadata dict each: `{"model": "X" or "Y", "year": 2023 to 2026}`.
2. Write `search_filtered(query, k, where)` from scratch: build a boolean mask from `where` (equality on `model`, minimum on `year`), score only the masked rows with the dot product, and return the top `k` texts.
3. Load the same notes into a Chroma collection with the same metadata and run the equivalent `where` query (Chroma uses `{"$and": [{"model": "X"}, {"year": {"$gte": 2025}}]}`).
4. You are done when both return the same top 3 for three queries, and you can say what happens to a pre-filtered search when the filter leaves only a handful of rows (hint: it becomes exact and cheap, which is why "filter first" is the default for selective filters).

#### Solution: a solution

In [ ]:
# Downloads a model or data on first run
import chromadb
import numpy as np

notes = [
    ("Model X belt replaced; tighten mount screws to 0.8 N·m.", {"model": "X", "year": 2024}),
    ("Model X desiccant changed at the coastal site.", {"model": "X", "year": 2025}),
    ("Model Y modem firmware updated to fix upload stalls.", {"model": "Y", "year": 2025}),
    ("Model X cooler fan replaced after noisy spectra.", {"model": "X", "year": 2026}),
    ("Model Y GPS antenna moved for a clear sky view.", {"model": "Y", "year": 2023}),
    ("Model Y belt slipping; wheel position errors logged.", {"model": "Y", "year": 2026}),
    ("Model X calibration lamp check, factors within 1%.", {"model": "X", "year": 2023}),
    ("Model X filter wheel home sensor cable repaired.", {"model": "X", "year": 2026}),
]
texts = [t for t, _ in notes]
metas = [m for _, m in notes]
V = model.encode(texts, normalize_embeddings=True)

def search_filtered(query, k=3, model_is=None, year_min=None):
    mask = np.ones(len(texts), dtype=bool)
    if model_is is not None:
        mask &= np.array([m["model"] == model_is for m in metas])
    if year_min is not None:
        mask &= np.array([m["year"] >= year_min for m in metas])
    q = model.encode([query], normalize_embeddings=True)[0]
    scores = np.where(mask, V @ q, -np.inf)                 # masked rows can never win
    top = [i for i in np.argsort(-scores)[:k] if mask[i]]
    return [texts[i] for i in top]

col = chromadb.Client().get_or_create_collection("notes", metadata={"hnsw:space": "cosine"})
col.add(ids=[f"n{i}" for i in range(len(texts))], documents=texts, embeddings=V.tolist(), metadatas=metas)

for query in ["belt problems", "fan noise", "firmware"]:
    mine = search_filtered(query, model_is="X", year_min=2025)
    theirs = col.query(query_embeddings=model.encode([query], normalize_embeddings=True).tolist(), n_results=3,
                       where={"$and": [{"model": "X"}, {"year": {"$gte": 2025}}]})["documents"][0]
    print(query, "| same:", mine == theirs)

Both agree because, with a small collection, Chroma's filtered search is exact too. The interesting case is scale: a selective filter leaves few candidates, so the database can scan them exactly and skip the ANN index entirely; a loose filter is applied during the graph search instead. Knowing which one your database does explains many "why is this query slow" tickets.

## How approximate nearest-neighbour search works

### Measure it yourself

In [ ]:
import numpy as np, time, faiss

rng = np.random.default_rng(0)
n, d = 50_000, 384
# real embeddings have structure: simulate 200 "topics" in a lower-dimensional subspace
centers = rng.normal(size=(200, 48))
Z = centers[rng.integers(0, 200, n)] + 0.6 * rng.normal(size=(n, 48))
X = (Z @ rng.normal(size=(48, d))).astype("float32")
X /= np.linalg.norm(X, axis=1, keepdims=True)
Q = X[rng.choice(n, 200, replace=False)] + 0.05 * rng.normal(size=(200, d)).astype("float32")
Q /= np.linalg.norm(Q, axis=1, keepdims=True)

exact = np.argsort(-(Q @ X.T), axis=1)[:, :10]                 # ground truth top-10

hnsw = faiss.IndexHNSWFlat(d, 16, faiss.METRIC_INNER_PRODUCT)
hnsw.add(X)
for ef in [16, 64, 256]:
    hnsw.hnsw.efSearch = ef
    t = time.perf_counter(); _, I = hnsw.search(Q, 10); dt = time.perf_counter() - t
    recall = np.mean([len(set(I[i]) & set(exact[i])) / 10 for i in range(len(Q))])
    print(f"ef_search={ef:<4} recall@10={recall:.3f}  {dt * 1000 / len(Q):.3f} ms/query")

```text
ef_search=16   recall@10=0.936  0.032 ms/query
ef_search=64   recall@10=0.994  0.062 ms/query
ef_search=256  recall@10=0.999  0.151 ms/query
```

## Clustering and visualising embeddings

Embeddings also let you explore collections you have never read. Embed a few thousand support tickets, cluster them with k-means (lesson 04), and ask an LLM to name each cluster from a sample of its members. You get an instant map of what users complain about.

In [ ]:
# Downloads a model or data on first run
tickets = [
    "Fan in the enclosure is grinding", "Cooler fan very loud since Tuesday", "Fan blade cracked",
    "GUI crashes after update", "Driver fails to load on startup", "App freezes on export",
    "No data since midnight", "Values all zero yesterday", "Upload failed again overnight",
    "Move the calibration visit to March", "Resend the invoice please", "Who handles site access?",
]
from sklearn.cluster import KMeans

ticket_vecs = model.encode(tickets, normalize_embeddings=True)
km = KMeans(n_clusters=4, n_init=10, random_state=0).fit(ticket_vecs)
for c in range(4):
    members = [t for t, lab in zip(tickets, km.labels_) if lab == c][:8]
    # name = llm("Give a 3-5 word label for these support tickets:\n" + "\n".join(members))
    print(c, len([1 for lab in km.labels_ if lab == c]), members[0][:80])

### Hands-on: a search engine for your own notes

1. Collect at least 200 paragraphs of text you know well: your learning log, documentation you use, or public-domain articles. Split into paragraphs.
2. Embed them with `all-MiniLM-L6-v2` and build a FAISS index.
3. Write 15 test queries, and for each note which paragraph should be found. Measure **recall@5** (the fraction of queries whose correct paragraph is in the top 5).
4. Compare with a second embedding model (for example `BAAI/bge-small-en-v1.5`, which expects the query prefix "Represent this sentence for searching relevant passages: "). Which wins on *your* data?
5. Find a query where keyword search would beat embeddings, and one where embeddings clearly win.

#### Solution: a solution sketch

In [ ]:
# Uses your own files: edit the paths or data first; Downloads a model or data on first run
import faiss, numpy as np
from sentence_transformers import SentenceTransformer

paras = [p.strip() for p in open("notes.txt").read().split("\n\n") if len(p.strip()) > 40]
tests = [("how do I stop overfitting", 17), ("what does the KV cache store", 88)]   # (query, correct index)

def evaluate(model_name, query_prefix=""):
    m = SentenceTransformer(model_name)
    D = m.encode(paras, normalize_embeddings=True).astype("float32")
    index = faiss.IndexFlatIP(D.shape[1]); index.add(D)
    hits = 0
    for q, gold in tests:
        qv = m.encode([query_prefix + q], normalize_embeddings=True).astype("float32")
        _, ids = index.search(qv, 5)
        hits += gold in ids[0]
    return hits / len(tests)

print(evaluate("all-MiniLM-L6-v2"))
print(evaluate("BAAI/bge-small-en-v1.5",
               query_prefix="Represent this sentence for searching relevant passages: "))

Typical findings: both models do well on paraphrased questions; the better one depends on your text, which is the point of measuring. Keyword search wins on exact strings such as error codes, version numbers and names; embeddings win on questions phrased differently from the text ("why is my model memorising the training data" finding a paragraph about overfitting). This small labelled query set is your first **retrieval evaluation**; you will grow it in lessons 14 and 17.

## Check yourself

Take the quiz at the end of [the lesson page](../../lessons/13-embeddings-vector-search.html). Then close everything and write three or four sentences in your learning log about what you learned, from memory.